<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Visión por Computadora 3D**
# **Clase 4, El flujo de Structure from Motion**

Versión adaptada para usar **tu propio video**.

programamos un SfM desde cero, caja por caja, y miramos qué produce cada paso. Al final repetimos los pasos 3 a 5 foto por foto y vemos crecer el modelo en una animación 3D

**no hace falta GPU**

---
subí tu video (mp4, mov...). En Colab se abre un selector de archivos; en Jupyter local poné la ruta en `RUTA_VIDEO`

In [ ]:
!pip install -q pycolmap==4.2.1

RUTA_VIDEO = None   # en Jupyter local: RUTA_VIDEO = "mi_video.mp4"
if RUTA_VIDEO is None:
    from google.colab import files
    RUTA_VIDEO = list(files.upload().keys())[0]
print("video:", RUTA_VIDEO)

importamos lo necesario

In [ ]:
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.optimize import least_squares
from scipy.sparse import lil_matrix
from scipy.spatial.transform import Rotation

**0.- Las fotos**

* la cámara se mueve alrededor de la escena (tu video)
* cortamos el video en N tramos y en cada uno nos quedamos con el cuadro más nítido (varianza del Laplaciano)

In [ ]:
N = 24            # cuántas fotos sacamos del video (más fotos = más lento, pero modelo más completo)
ANCHO_MAX = 1280  # achicamos los cuadros grandes (4K/1080p) para que SIFT y el BA no tarden una eternidad
cap = cv2.VideoCapture(RUTA_VIDEO)
video = []
while True:
    ok, fr = cap.read()
    if not ok: break
    if fr.shape[1] > ANCHO_MAX:
        fr = cv2.resize(fr, (ANCHO_MAX, round(fr.shape[0] * ANCHO_MAX / fr.shape[1])), interpolation=cv2.INTER_AREA)
    video.append(fr)

# calculamos qué tan nítido es cada cuadro con la varianza del Laplaciano
# el Laplaciano resalta los bordes: una foto nítida tiene bordes marcados, así que la varianza da alta
# una foto movida o borrosa da baja
nitidez = [cv2.Laplacian(cv2.cvtColor(c, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var() for c in video]

# dividimos el video en 24 tramos y de cada tramo nos quedamos con el cuadro más nítido
elegidos = [t[np.argmax([nitidez[i] for i in t])] for t in np.array_split(np.arange(len(video)), N)]

# convertimos las fotos de BGR (el orden de colores de OpenCV) a RGB para mostrarlas
fotos = [cv2.cvtColor(video[k], cv2.COLOR_BGR2RGB) for k in elegidos]

# tambien generamos versiones en gris, porque SIFT trabaja en blanco y negro
grises = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in fotos]
H, W = grises[0].shape
print(f"{len(video)} cuadros en el video -> {N} fotos de {W} × {H}")

fig, ax = plt.subplots(int(np.ceil(N / 8)), 8, figsize=(20, 1.7 * np.ceil(N / 8)))
for k, a in enumerate(ax.ravel()):
    a.axis("off")
    if k < N: a.imshow(fotos[k]); a.set_title(f"foto {k}", fontsize=9)
plt.show()

**1.- Features**

puntos distintivos en cada foto

SIFT busca en cada foto puntos que se puedan volver a encontrar desde otro lugar: esquinas y manchas, a la escala en que aparecen.

Cada uno tiene una posición, una escala, una orientación y un descriptor de 128 números

In [ ]:
# buscamos hasta 3000 puntos característicos por foto: esquinas o manchas que se
# pueden reconocer aunque la foto se haya tomado desde otro lugar
sift = cv2.SIFT_create(3000)

# para cada foto devuelve:
# kps (keypoints): posición, escala (tamaño) y orientación de cada punto
# descs: un descriptor de 128 números por punto, funciona como una “huella digital” del punto
kps, descs = zip(*[sift.detectAndCompute(g, None) for g in grises])

# xy guarda solo las coordenadas (u, v) en píxeles
xy = [np.float64([k.pt for k in ks]) for ks in kps]
print(f"features por foto: entre {min(map(len, kps))} y {max(map(len, kps))}")

los features de la primera foto: el círculo es la escala y la línea la orientación

se concentran donde hay textura (los arcos, los edificios) y casi no hay en el cielo o el asfalto

In [ ]:
vis = cv2.drawKeypoints(fotos[0], kps[0][::4], None, color=(255, 230, 0), flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
plt.figure(figsize=(14, 8))
plt.imshow(vis)
plt.axis("off")
plt.title("1 de cada 4 features SIFT de la foto 0")
plt.show()

**2.- Matching**

el mismo punto en dos fotos

para cada par de fotos:
* para cada descriptor de una foto buscamos los 2 más parecidos en la otra
* **test de Lowe:** aceptamos si el mejor es claramente mejor que el segundo (distancia < 0.8 × la segunda)
* **RANSAC** sobre la matriz fundamental: descartamos las parejas que no cumplen x₂ᵀ F x₁ = 0

como es un video, comparamos cada foto con las 10 siguientes

In [ ]:
bf = cv2.BFMatcher()
def emparejar(i, j):
    # Buscamos los 2 descriptores más parecidos y aplicamos el test de Lowe
    ms = [m for m, n in bf.knnMatch(descs[i], descs[j], k=2) if m.distance < 0.8 * n.distance]
    a = np.array([m.queryIdx for m in ms])
    b = np.array([m.trainIdx for m in ms])

    if len(ms) < 20:
      return None

    # Estimamos la matriz fundamental F con RANSAC y descartamos las parejas que no cumplen la geometría
    F, ok = cv2.findFundamentalMat(xy[i][a], xy[j][b], cv2.FM_RANSAC, 1.0, 0.999)
    if F is None:
      return None
    ok = ok.ravel() > 0
    return {"a": a, "b": b, "ok": ok, "F": F}


VENTANA = 10  # con cuántas fotos siguientes comparamos cada una (subilo a N si tu video vuelve al punto de partida)
t0 = time.time()
pares = {}
for i in range(N):
    # Comparamos cada foto con las VENTANA siguientes
    for j in range(i + 1, min(i + 1 + VENTANA, N)):
        r = emparejar(i, j)
        # Guardamos el par solo si tiene al menos 30 parejas verificadas
        if r is not None and r["ok"].sum() >= 30:
          pares[(i, j)] = r
print(f"{len(pares)} pares con al menos 30 parejas verificadas | {time.time() - t0:.1f} s")

un par de fotos: en verde las parejas que pasaron RANSAC, en rojo las que pasaron Lowe pero no la geometría

In [ ]:
def dibujar_par(i, j, n=120):
    r = pares[(i, j)]
    # Elegimos al azar hasta 120 parejas para no saturar el dibujo
    sel = np.random.default_rng(0).choice(len(r["a"]), min(n, len(r["a"])), replace=False)
    # Ponemos las dos fotos lado a lado
    plt.figure(figsize=(18, 5)); plt.imshow(np.hstack([fotos[i], fotos[j]])); plt.axis("off")
    for k in sel:
        p, q = xy[i][r["a"][k]], xy[j][r["b"][k]]
        # Unimos cada pareja con una línea: verde si pasó RANSAC, roja si no (sumamos W porque la segunda foto está a la derecha)
        plt.plot([p[0], q[0] + W], [p[1], q[1]], color="lime" if r["ok"][k] else "red", lw=0.8)
    plt.title(f"fotos {i} y {j}: {r['ok'].sum()} de {len(r['a'])} parejas verificadas")
    plt.show()

# dibujamos el par con más parejas verificadas entre fotos separadas por al menos 2 cuadros
par_demo = max([k for k in pares if k[1] - k[0] >= 2] or list(pares), key=lambda k: pares[k]["ok"].sum())
dibujar_par(*par_demo)

cuántas parejas verificadas hay entre cada par de fotos. Las fotos cercanas en el video comparten mucho; a medida que el dron gira, cada vez menos

In [ ]:
M = np.zeros((N, N))
# Llenamos la matriz con la cantidad de parejas verificadas de cada par (es simétrica)
for (i, j), r in pares.items(): M[i, j] = M[j, i] = r["ok"].sum()
# Mostramos la matriz como imagen: más oscuro = más parejas en común
plt.figure(figsize=(6, 5)); plt.imshow(M, cmap="Blues"); plt.colorbar(label="parejas verificadas")
plt.xlabel("foto")
plt.ylabel("foto")
plt.title("matriz de matching")
plt.show()

**tracks:** encadenamos las parejas. Si el feature 12 de la foto 0 es el 40 de la foto 1, y ese es el 7 de la foto 2, los tres son el **mismo punto 3D**. Lo hacemos con union-find: cada feature arranca en su propio grupo y cada pareja une dos grupos

In [ ]:
padre = {}
# Buscamos la raíz del grupo de un nodo y acortamos el camino para que sea más rápido
def raiz(n):
    while padre.setdefault(n, n) != n:
        padre[n] = padre[padre[n]]
        n = padre[n]
    return n

# Unimos en un mismo grupo cada pareja verificada (un nodo es (foto, índice del feature))
for (i, j), r in pares.items():
    for p, q in zip(r["a"][r["ok"]], r["b"][r["ok"]]):
        padre[raiz((i, p))] = raiz((j, q))

grupos = {}
for n in list(padre):
  grupos.setdefault(raiz(n), []).append(n)

# Nos quedamos con los grupos que no tienen dos features de la misma foto: cada uno es un track
tracks = [sorted(g) for g in grupos.values() if len(g) >= 2 and len({f for f, _ in g}) == len(g)]

# Armamos el índice inverso: para cada foto, de feature a track
track_de = [dict() for _ in range(N)]
for t, g in enumerate(tracks):
    for f, k in g:
      track_de[f][k] = t

largos = np.array([len(t) for t in tracks])

print(f"{len(tracks)} tracks | largo medio {largos.mean():.1f} fotos | el más largo: {largos.max()} fotos")
# Graficamos en cuántas fotos aparece cada track
plt.figure(figsize=(8, 3))
plt.hist(largos, bins=np.arange(2, largos.max() + 2) - 0.5)
plt.xlabel("en cuántas fotos aparece")
plt.ylabel("tracks")
plt.yscale("log")
plt.show()

**un punto, muchas fotos:** el track más largo. Es el mismo lugar del mundo visto desde todo el recorrido del dron

In [ ]:
# Buscamos el track más largo
t = int(np.argmax(largos))
# Tomamos hasta 12 de sus observaciones, repartidas a lo largo del recorrido
obs = tracks[t][:: max(1, len(tracks[t]) // 12)][:12]
fig, ax = plt.subplots(2, len(obs), figsize=(2 * len(obs), 3.2))
for k, (f, i) in enumerate(obs):
    u, v = xy[f][i]
    # Arriba: marcamos el punto con un círculo rojo en la foto completa
    ax[0, k].imshow(fotos[f]); ax[0, k].scatter(u, v, s=60, facecolors="none", edgecolors="red", lw=2)
    # Abajo: hacemos zoom recortando 40 px alrededor del punto
    ax[1, k].imshow(fotos[f][max(0, int(v) - 40):int(v) + 40, max(0, int(u) - 40):int(u) + 40])
    ax[0, k].set_title(f"foto {f}", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.show()

## **3.- Geometría de dos vistas**

E, R y t entre cámaras

* arrancamos con un par de fotos

* tiene que tener muchas parejas, pero también buena separación: con fotos casi iguales los rayos son paralelos y la triangulación es mala

* no conocemos K

* suponemos el centro óptico en el medio de la imagen, la focal la vemos enseguida

In [ ]:
# Armamos la matriz de calibración K con la focal f y el centro en el medio de la imagen
def K_de(f):
    return np.array([[f, 0, W / 2], [0, f, H / 2], [0, 0, 1]])

# Triangulamos con DLT: cada observación aporta 2 filas a A y el punto 3D es el último vector singular
def triangular(Ps, uvs):
    A = np.concatenate([np.stack([uv[:, :1] * P[2] - P[0], uv[:, 1:] * P[2] - P[1]], 1) for P, uv in zip(Ps, uvs)], 1)
    X = np.linalg.svd(A)[2][:, -1]
    return X[:, :3] / X[:, 3:]

# Proyectamos puntos 3D en la cámara P: devolvemos el píxel y la profundidad
def proyectar_P(P, X):
    h = np.c_[X, np.ones(len(X))] @ P.T
    return h[:, :2] / h[:, 2:], h[:, 2]

def dos_vistas(i, j, f):
    r = pares[(i, j)]
    p, q = xy[i][r["a"][r["ok"]]], xy[j][r["b"][r["ok"]]]
    K = K_de(f)
    # Calculamos E = Kᵀ F K y recuperamos la pose (R, t) que deja los puntos delante de las cámaras
    _, R, t, _ = cv2.recoverPose(K.T @ r["F"] @ K, p, q, K)
    # Ubicamos la primera cámara en el origen y la segunda con (R, t)
    P1, P2 = K @ np.c_[np.eye(3), np.zeros(3)], K @ np.c_[R, t]
    X = triangular([P1, P2], [p, q])
    # Medimos el error de reproyección promedio entre las dos fotos
    e = (np.linalg.norm(proyectar_P(P1, X)[0] - p, axis=1) + np.linalg.norm(proyectar_P(P2, X)[0] - q, axis=1)) / 2
    # Calculamos el ángulo entre los rayos de las dos cámaras hacia cada punto
    C2 = -R.T @ t.ravel()
    cos = (X * (X - C2)).sum(1) / np.linalg.norm(X, axis=1) / np.linalg.norm(X - C2, axis=1)
    return R, t.ravel(), np.median(e), np.degrees(np.median(np.arccos(np.clip(cos, -1, 1))))

las rectas epipolares de ese par: cada punto de la foto izquierda tiene que tener su pareja sobre la recta del mismo color

In [ ]:
f_ini = 1.2 * max(W, H)

f0 = 1.2 * W
candidatos = []
for (i, j), r in pares.items():
    if 4 <= j - i <= 10:
        _, _, _, ang = dos_vistas(i, j, f0)
        candidatos.append((r["ok"].sum() if ang > 3 else 0, i, j, ang))
# si tu video no tiene pares separados entre 4 y 10 fotos (pocas fotos, o mucho movimiento), probamos con todos
if not candidatos:
    for (i, j), r in pares.items():
        _, _, _, ang = dos_vistas(i, j, f0)
        candidatos.append((r["ok"].sum() if ang > 3 else 0, i, j, ang))
_, A0, B0, ang = max(candidatos)
print(f"par inicial: fotos {A0} y {B0} | {pares[(A0, B0)]['ok'].sum()} parejas | ángulo mediano {ang:.1f}°")

focales = np.arange(0.4, 2.01, 0.05) * W  # rango relativo al ancho, así sirve para cualquier resolución
errores = [dos_vistas(A0, B0, f)[2] for f in focales]

r = pares[(A0, B0)]
# Tomamos las parejas verificadas del par inicial
p, q = xy[A0][r["a"][r["ok"]]], xy[B0][r["b"][r["ok"]]]
# Elegimos 10 parejas repartidas para dibujar
sel = np.linspace(0, len(p) - 1, 10).astype(int)
fig, ax = plt.subplots(1, 2, figsize=(18, 5))
ax[0].imshow(fotos[A0]); ax[1].imshow(fotos[B0])
for c, k in zip(plt.cm.tab10(np.arange(10)), sel):
    # Calculamos la recta epipolar l = F·p del punto de la foto izquierda
    l = r["F"] @ [*p[k], 1]; u = np.array([0, W])
    # Dibujamos la recta en la foto derecha despejando v de la ecuación de la recta
    ax[1].plot(u, -(l[0] * u + l[2]) / l[1], color=c, lw=1.3)
    # Marcamos el punto y su pareja con el mismo color: la pareja debe caer sobre la recta
    ax[0].scatter(*p[k], color=c, s=50)
    ax[1].scatter(*q[k], color=c, s=50)
for a, tt in zip(ax, [f"foto {A0}", f"foto {B0}: rectas epipolares"]):
    a.set_xlim(0, W)
    a.set_ylim(H, 0)
    a.axis("off")
    a.set_title(tt)
plt.show()

E = Kᵀ F K tiene **4** descomposiciones posibles en (R, t). Solo una deja los puntos delante de las dos cámaras

In [ ]:
K = K_de(f_ini)

# Descomponemos E en sus 2 rotaciones posibles y la dirección de t
R1, R2, tt = cv2.decomposeEssentialMat(K.T @ r["F"] @ K)
P1 = K @ np.c_[np.eye(3), np.zeros(3)]
nombres, delante = [], []

# Probamos las 4 combinaciones de (R, t)
for nombre, R, t in [("R₁, +t", R1, tt), ("R₁, −t", R1, -tt), ("R₂, +t", R2, tt), ("R₂, −t", R2, -tt)]:
    X = triangular([P1, K @ np.c_[R, t]], [p, q])
    # Contamos qué porcentaje de puntos queda delante de las dos cámaras (z > 0 en ambas)
    nombres.append(nombre)
    delante.append(((X[:, 2] > 0) & ((X @ R.T + t.T)[:, 2] > 0)).mean() * 100)

# Graficamos y resaltamos en verde la única solución válida
plt.figure(figsize=(7, 3))
plt.bar(nombres, delante, color=["tab:green" if d == max(delante) else "tab:gray" for d in delante])
plt.ylabel("% de puntos delante de las 2 cámaras")
plt.title("las 4 soluciones de E"); plt.show()

# Calculamos la pose inicial con la focal elegida y mostramos cuánto giró la segunda cámara
R_ini, t_ini, e_ini, _ = dos_vistas(A0, B0, f_ini)
print(f"la cámara {B0} giró {np.degrees(np.arccos((np.trace(R_ini) - 1) / 2)):.1f}° | t = {t_ini.round(3)} (norma 1: sin escala)")

**4.- Triangulación**


los puntos 3D

el estado de nuestra reconstrucción:
* **camaras:** para cada foto registrada, su pose como 6 números (rotación de Rodrigues + traslación)
* **puntos:** para cada track triangulado, su posición 3D
* **f:** la focal, compartida por todas las fotos

usamos la cámara de la primera foto (A0) como referencia para definir las coordenadas 3D

In [ ]:
# Guardamos cada cámara como 6 números (rotación + traslación): A0 queda en el origen
camaras = {A0: np.zeros(6), B0: np.r_[Rotation.from_matrix(R_ini).as_rotvec(), t_ini]}
puntos = {} # Guardamos aquí la posición 3D de cada track triangulado
f = f_ini # Usamos una sola focal para todas las fotos

# Armamos la matriz de proyección P = K [R | t] a partir de los 6 números
def P_de(c, f):
    return K_de(f) @ np.c_[Rotation.from_rotvec(c[:3]).as_matrix(), c[3:]]

# Calculamos dónde está la cámara en el mundo: C = −Rᵀ t
def centro(c):
    return -Rotation.from_rotvec(c[:3]).as_matrix().T @ c[3:]

Reconstruimos nuevos puntos 3D a partir de las correspondencias (tracks) observadas en al menos dos cámaras registradas:

1. Busca candidatos: identifica tracks que todavía no tienen coordenadas 3D y aparecen en al menos dos cámaras
2. Triangula: utiliza las matrices de proyección de las cámaras y DLT para calcular la posición 3D de cada punto
3. Valida los resultados: descarta puntos que estén detrás de alguna cámara o tengan un error de reproyección superior a 4 píxeles
4. Comprueba la geometría: exige un ángulo de triangulación superior a 1.5° para evitar reconstrucciones poco fiables
5. Guarda los puntos válidos: almacena sus coordenadas 3D en puntos

In [ ]:
def triangular_nuevos():
    # Buscamos los tracks sin punto 3D que aparecen en al menos 2 cámaras registradas
    cand = [t for t, g in enumerate(tracks) if t not in puntos and sum(fr in camaras for fr, _ in g) >= 2]
    nuevos = 0
    # Agrupamos por cantidad de observaciones para triangular muchos a la vez
    for n_obs in {sum(fr in camaras for fr, _ in tracks[t]) for t in cand}:
        ts = [t for t in cand if sum(fr in camaras for fr, _ in tracks[t]) == n_obs]
        obs = [[(fr, k) for fr, k in tracks[t] if fr in camaras] for t in ts]
        Ps = [[P_de(camaras[fr], f) for fr, _ in o] for o in obs]
        # Triangulamos con DLT: armamos A para cada track y resolvemos todo con una sola SVD
        A = np.array([np.concatenate([[x[0] * P[2] - P[0], x[1] * P[2] - P[1]] for P, (fr, k) in zip(Pl, o) for x in [xy[fr][k]]]) for Pl, o in zip(Ps, obs)])
        Xh = np.linalg.svd(A)[2][:, -1]; X = Xh[:, :3] / Xh[:, 3:]
        for t, o, x in zip(ts, obs, X):
            ok, rayos = True, []
            for fr, k in o:
                uv, z = proyectar_P(P_de(camaras[fr], f), x[None])
                # Descartamos el punto si queda detrás de una cámara o tiene más de 4 px de error
                if z[0] <= 0 or np.linalg.norm(uv[0] - xy[fr][k]) > 4: ok = False; break
                rayos.append((x - centro(camaras[fr])) / np.linalg.norm(x - centro(camaras[fr])))
            # Aceptamos el punto solo si el ángulo máximo entre rayos supera 1.5°
            if ok and np.degrees(np.arccos(np.clip(min(a @ b for a in rayos for b in rayos), -1, 1))) > 1.5:
                puntos[t] = x; nuevos += 1
    return nuevos

print(f"puntos triangulados con 2 fotos: {triangular_nuevos()}")

vemos los puntos
* obtenemos los colores directamente de los píxeles de las fotografías originales

* para cada track toma el color del píxel correspondiente a su primera observación (g[0]) en la primera fotografía donde aparece

* cada punto 3D se visualiza con un color similar al que tenía en la imagen original
* las cámaras obtienen sus coordenadas a partir de sus parámetros de rotación y traslación, almacenados en el diccionario camaras

In [ ]:
# Tomamos el color de cada track del píxel de la primera foto donde aparece
color_track = np.array([fotos[g[0][0]][int(xy[g[0][0]][g[0][1]][1]), int(xy[g[0][0]][g[0][1]][0])] for g in tracks]) / 255

# Dibujamos cada cámara como una pirámide: el vértice es el centro óptico y la base es la imagen
def piramide(c, f, tam):
    R = Rotation.from_rotvec(c[:3]).as_matrix(); C = centro(c)
    # Proyectamos las 4 esquinas de la imagen hacia adelante y las pasamos al sistema del mundo
    esquinas = np.array([[0, 0, 1], [W, 0, 1], [W, H, 1], [0, H, 1]]) @ np.linalg.inv(K_de(f)).T * tam
    E = esquinas @ R + C
    L = [C, E[0], None, C, E[1], None, C, E[2], None, C, E[3], None, E[0], E[1], E[2], E[3], E[0], None]
    return [(np.full(3, np.nan) if p is None else p) for p in L]

# Reordenamos los ejes a (X, Z, −Y) para que "arriba" quede hacia arriba en el gráfico
def a_vista(P):
    P = np.asarray(P); return P[:, 0], P[:, 2], -P[:, 1]

def figura(camaras, puntos, f, titulo, tam=None, max_puntos=6000):
    ts = np.array(sorted(puntos)); X = np.array([puntos[t] for t in ts])
    # Limitamos la cantidad de puntos para que el gráfico no se vuelva lento
    if len(ts) > max_puntos:
        k = np.random.default_rng(0).choice(len(ts), max_puntos, replace=False); ts, X = ts[k], X[k]
    tam = tam or 0.15 * np.median(np.linalg.norm(X - centro(camaras[A0]), axis=1))
    lineas = [p for c in camaras.values() for p in piramide(c, f, tam)]
    col = [f"rgb({r * 255:.0f},{g * 255:.0f},{b * 255:.0f})" for r, g, b in color_track[ts]]
    # Devolvemos los puntos con su color y las cámaras en rojo
    return [go.Scatter3d(x=a_vista(X)[0], y=a_vista(X)[1], z=a_vista(X)[2], mode="markers", marker=dict(size=1.5, color=col), name="puntos"),
            go.Scatter3d(x=a_vista(lineas)[0], y=a_vista(lineas)[1], z=a_vista(lineas)[2], mode="lines", line=dict(color="crimson", width=3), name="cámaras")]

fig = go.Figure(figura(camaras, puntos, f, ""))
fig.update_layout(height=550, title=f"2 cámaras y {len(puntos)} puntos", scene=dict(aspectmode="data", xaxis_visible=False, yaxis_visible=False, zaxis_visible=False), margin=dict(l=0, r=0, t=40, b=0))
fig.show()

**5.- Bundle Adjustment**


refinar todo junto

ajustamos **todo junto**: las poses de las cámaras registradas, los puntos y la focal, para minimizar el error de reproyección

Σ ‖ xᵢⱼ − π(f, Rᵢ, tᵢ, Xⱼ) ‖²

* la cámara A0 queda fija: es el origen del mundo
* el jacobiano es disperso: cada residuo depende de su cámara, su punto y la focal (notebook 2)
* las parejas malas que se colaron no las ablandamos con una pérdida robusta: después de cada ajuste borramos los puntos con alguna observación de más de 4 px

In [ ]:
# Listamos todas las observaciones: qué cámara (ic), qué punto (ip) y dónde se midió (uv)
def observaciones():
    libres = [fr for fr in sorted(camaras) if fr != A0] # Ajustamos todas las cámaras menos A0
    ts = sorted(puntos)
    ic, ip, uv = [], [], []
    for n, t in enumerate(ts):
        for fr, k in tracks[t]:
            if fr in camaras:
                ic.append(-1 if fr == A0 else libres.index(fr)); ip.append(n); uv.append(xy[fr][k])
    return libres, ts, np.array(ic), np.array(ip), np.array(uv)

# Proyectamos todos los puntos en sus cámaras de una sola vez
def proyectar(cam, f, X, ic, ip):
    cam = np.r_[cam, np.zeros((1, 6))] # Usamos el índice −1 para la cámara fija A0
    Xc = Rotation.from_rotvec(cam[ic, :3]).apply(X[ip]) + cam[ic, 3:]
    return f * Xc[:, :2] / Xc[:, 2:] + [W / 2, H / 2]

# Medimos la distancia en píxeles entre lo proyectado y lo medido
def error_reproyeccion():
    libres, ts, ic, ip, uv = observaciones()
    cam = np.array([camaras[fr] for fr in libres]).reshape(-1, 6); X = np.array([puntos[t] for t in ts])
    return np.linalg.norm(proyectar(cam, f, X, ic, ip) - uv, axis=1)

def bundle_adjustment(max_iter=20, fijar_f=False):
    global f
    libres, ts, ic, ip, uv = observaciones()
    nc, npt, nf = len(libres), len(ts), 0 if fijar_f else 1
    # Definimos los residuos: la diferencia entre la proyección y la medición
    def residuos(v):
        ff = f if fijar_f else v[6 * nc]
        return (proyectar(v[:6 * nc].reshape(nc, 6), ff, v[6 * nc + nf:].reshape(npt, 3), ic, ip) - uv).ravel()
    # Armamos la matriz dispersa: cada residuo depende solo de su cámara, su punto y la focal
    A = lil_matrix((2 * len(uv), 6 * nc + nf + 3 * npt), dtype=int)
    filas = np.arange(len(uv)); libre = ic >= 0
    for s in range(6):
        A[2 * filas[libre], 6 * ic[libre] + s] = 1; A[2 * filas[libre] + 1, 6 * ic[libre] + s] = 1
    if nf: A[:, 6 * nc] = 1
    for s in range(3):
        A[2 * filas, 6 * nc + nf + 3 * ip + s] = 1; A[2 * filas + 1, 6 * nc + nf + 3 * ip + s] = 1
    # Juntamos todas las variables en un vector: cámaras, focal y puntos
    v0 = np.r_[np.array([camaras[fr] for fr in libres]).ravel(), [f] * nf, np.array([puntos[t] for t in ts]).ravel()]
    # Optimizamos todo junto por mínimos cuadrados
    sol = least_squares(residuos, v0, jac_sparsity=A, method="trf", x_scale="jac", max_nfev=max_iter)
    # Guardamos los valores ajustados
    for k, fr in enumerate(libres): camaras[fr] = sol.x[6 * k:6 * k + 6]
    if nf: f = sol.x[6 * nc]
    for k, t in enumerate(ts): puntos[t] = sol.x[6 * nc + nf + 3 * k:6 * nc + nf + 3 * k + 3]

# Borramos los puntos con alguna observación de más de `umbral` px
def limpiar(umbral=4):
    libres, ts, ic, ip, uv = observaciones()
    e = error_reproyeccion()
    malos = np.unique(ip[e > umbral])
    for n in malos: del puntos[ts[n]]
    return len(malos)

con solo 2 fotos ya hay algo para ajustar

In [ ]:
e0 = error_reproyeccion().mean() # Medimos el error antes de ajustar
bundle_adjustment() # Ajustamos cámaras, puntos y focal con solo 2 fotos
# Comparamos el error y la focal antes y después del ajuste
print(f"error de reproyección: {e0:.2f} px -> {error_reproyeccion().mean():.2f} px | focal {f_ini:.0f} -> {f:.0f} px")

**SfM incremental**

ahora repetimos foto por foto:
1. elegimos la foto no registrada que ve más puntos 3D ya triangulados
2. estimamos la pose de la nueva cámara utilizando puntos 3D conocidos y sus correspondencias 2D en la nueva fotografía
3. triangulamos los tracks nuevos que ahora ven 2 fotos registradas
4. bundle adjustment de todo, y limpiamos puntos malos

guardamos una foto del estado en cada paso para animarlo

In [ ]:
def registrar_siguiente():
    K = K_de(f)
    mejor = None
    # Buscamos la foto no registrada que ve más puntos 3D ya triangulados
    for fr in range(N):
        if fr in camaras:
          continue
        pares2d3d = [(k, t) for k, t in track_de[fr].items() if t in puntos]
        if mejor is None or len(pares2d3d) > len(mejor[1]):
          mejor = (fr, pares2d3d)
    fr, p2d3d = mejor
    if len(p2d3d) < 30:
      return None
    obj = np.array([puntos[t] for _, t in p2d3d]); img = np.array([xy[fr][k] for k, _ in p2d3d])
    # Estimamos la pose de la foto nueva con PnP + RANSAC a partir de las parejas 2D–3D
    ok, rvec, tvec, inl = cv2.solvePnPRansac(obj, img, K, None, iterationsCount=500, reprojectionError=4.0)
    if not ok or inl is None or len(inl) < 30:
      return None
    camaras[fr] = np.r_[rvec.ravel(), tvec.ravel()] # Registramos la cámara nueva
    return fr, len(p2d3d), len(inl)

historia = [(dict(camaras), dict(puntos), f, "inicio")] # Guardamos cada estado para la animación
filas = []
t0 = time.time()
# Repetimos foto por foto hasta registrar todas
while len(camaras) < N:
    r = registrar_siguiente()
    if r is None:
      print("ninguna foto más se puede registrar")
      break
    fr, n2d3d, n_inl = r
    nuevos = triangular_nuevos() # Triangulamos los puntos que ahora ven 2 cámaras
    antes = error_reproyeccion().mean()
    bundle_adjustment(max_iter=10) # Ajustamos todo junto
    borrados = limpiar() # Borramos los puntos malos
    despues = error_reproyeccion().mean()
    filas.append((len(camaras), fr, n2d3d, n_inl, nuevos, len(puntos), antes, despues, f))
    historia.append((dict(camaras), dict(puntos), f, f"+ foto {fr}"))
    print(f"{len(camaras):2d} cámaras | + foto {fr:2d} | PnP: {n_inl:4d}/{n2d3d:4d} inliers | +{nuevos:4d} puntos | total {len(puntos):5d} | error {antes:.2f} -> {despues:.2f} px | f {f:.0f} px")
print(f"listo en {time.time() - t0:.0f} s")

**la animación:** cada cuadro es el estado después de agregar una foto

In [ ]:
# Calculamos límites fijos para los ejes, así la vista no salta entre cuadros
def ejes_de(camaras, puntos):
    X = np.array(list(puntos.values()) + [centro(c) for c in camaras.values()])
    # Usamos los percentiles 1 y 99 para ignorar puntos sueltos y agregamos un margen del 10 %
    lo, hi = np.percentile(X, 1, axis=0), np.percentile(X, 99, axis=0)
    pad = 0.1 * (hi - lo)
    lo, hi = lo - pad, hi + pad
    return dict(xaxis=dict(range=[lo[0], hi[0]], visible=False), yaxis=dict(range=[lo[2], hi[2]], visible=False),
                zaxis=dict(range=[-hi[1], -lo[1]], visible=False), aspectmode="manual",
                aspectratio=dict(x=1, y=(hi[2] - lo[2]) / (hi[0] - lo[0]), z=(hi[1] - lo[1]) / (hi[0] - lo[0])))

final_cam, final_pts = historia[-1][0], historia[-1][1]
# Fijamos el tamaño de las pirámides según el modelo final
tam = 0.12 * np.median(np.linalg.norm(np.array(list(final_pts.values())) - centro(final_cam[A0]), axis=1))
# Creamos un cuadro de animación por cada estado guardado en la historia
cuadros_anim = [go.Frame(data=figura(c, p, ff, "", tam, 4000), name=str(k), layout=go.Layout(title=f"{len(c)} cámaras, {len(p)} puntos ({nombre})"))
                for k, (c, p, ff, nombre) in enumerate(historia)]
fig = go.Figure(data=cuadros_anim[0].data, frames=cuadros_anim)
# Agregamos el botón de play y el slider para recorrer los pasos
fig.update_layout(height=650, scene=ejes_de(final_cam, final_pts), margin=dict(l=0, r=0, t=40, b=0), title=cuadros_anim[0].layout.title.text,
    updatemenus=[dict(type="buttons", x=0.05, y=0.05, buttons=[dict(label="▶ play", method="animate", args=[None, dict(frame=dict(duration=600, redraw=True), fromcurrent=True)])])],
    sliders=[dict(x=0.15, y=0.02, len=0.8, currentvalue=dict(prefix="paso "), steps=[dict(label=str(k), method="animate", args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for k in range(len(historia))])])
fig.show()


**7.- El resultado**

un último bundle adjustment largo (hasta 150 iteraciones), limpiando con un umbral más estricto (2 px)

In [ ]:
f_antes = f
t0 = time.time()
# Hacemos un ajuste largo, limpiamos con un umbral más estricto (2 px) y volvemos a ajustar
bundle_adjustment(max_iter=150); limpiar(2); bundle_adjustment(max_iter=30)
e = error_reproyeccion()
print(f"{time.time() - t0:.0f} s | {len(camaras)} cámaras | {len(puntos)} puntos | error medio {e.mean():.2f} px | focal {f_antes:.0f} -> {f:.0f} px")

In [ ]:
fig = go.Figure(figura(camaras, puntos, f, "", tam, 20000))
fig.update_layout(height=700, title="tu escena con nuestro SfM", scene=ejes_de(camaras, puntos), margin=dict(l=0, r=0, t=40, b=0))
fig.show()

cada punto del modelo, proyectado de vuelta sobre una foto: si la reconstrucción es buena, los puntos caen justo sobre los features que lo generaron

In [ ]:
fr = sorted(camaras)[len(camaras) // 2] # Elegimos una foto del medio del recorrido
ts = [t for t in puntos if fr in {g for g, _ in tracks[t]}] # Buscamos los puntos 3D que se ven en esa foto
# Proyectamos los puntos 3D sobre la foto
uv_proy, _ = proyectar_P(P_de(camaras[fr], f), np.array([puntos[t] for t in ts]))
# Tomamos dónde se midió cada feature originalmente
uv_med = np.array([xy[fr][dict(tracks[t])[fr]] for t in ts])
plt.figure(figsize=(14, 8)); plt.imshow(fotos[fr])
# Comparamos: si la reconstrucción es buena, los puntos rojos caen dentro de los círculos amarillos
plt.scatter(*uv_med.T, s=12, facecolors="none", edgecolors="yellow", lw=0.8, label="feature medido")
plt.scatter(*uv_proy.T, s=3, c="red", label="punto 3D proyectado")
plt.legend(loc="lower right"); plt.axis("off"); plt.title(f"foto {fr}: {len(ts)} puntos"); plt.show()